# Pacotes

In [3]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from prettytable import PrettyTable
import sklearn
from sklearn.metrics import confusion_matrix

import time
from tensorflow import keras
import tensorflow as tf
from keras.models import Sequential
from keras.models import Sequential
from keras.layers import Dense, LSTM, Bidirectional, Flatten #Global max pullin avg max pulling
from keras.layers import  Masking
from keras.regularizers import l2
from tensorflow.keras import regularizers
from keras import callbacks
from keras.callbacks import EarlyStopping, ModelCheckpoint

import datetime

tf.config.list_physical_devices('GPU')

# fix random seed for reproducibility
seed = 42
tf.random.set_seed(seed)

2023-06-01 09:30:42.567694: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-06-01 09:30:42.635834: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-06-01 09:30:42.652328: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-06-01 09:30:42.940433: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: li

In [2]:
! pip install prettytable

/bin/bash: /home/pdpa-chuvas/miniconda3/envs/tf/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


# Funcoes

In [4]:
##############################
###### Matriz de confusão ####
##############################

def matriz_confusao(y_real,y_predito,modelo):

### Grafico ###

  tabela=confusion_matrix(y_real,y_predito)

  group_names = ["True Neg","False Pos","False Neg","True Pos"]
  group_counts = ["{0:0.0f}".format(value) for value in
                tabela.flatten()]
  group_percentages = ["{0:.5%}".format(value) for value in
                     tabela.flatten()/np.sum(tabela)]
  labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
          zip(group_names,group_counts,group_percentages)]
  labels = np.asarray(labels).reshape(2,2)
  f = plt.figure()
  f.set_figwidth(8)
  f.set_figheight(8)

  sns.heatmap(tabela, annot=labels, fmt="", cmap='Blues')

### Tabela ###
  Resultados=PrettyTable()
  Resultados.field_names=["Métrica","Resultado"]
  Resultados.title= modelo
  Resultados.align["Métrica"]="l"
  Resultados.align["Resultado"]="r"

  Resultados.add_row(["Acurácia:",round(sklearn.metrics.accuracy_score(y_real,y_predito),2)])
  Resultados.add_row(["Precisão:",round(sklearn.metrics.precision_score(y_real,y_predito),2)])
  Resultados.add_row(["Recall:",round(sklearn.metrics.recall_score(y_real,y_predito),2)])
  Resultados.add_row(["F1-Score:",round(sklearn.metrics.f1_score(y_real,y_predito),2)])

  print(Resultados)
  
  return



    
def transformacao_estrutura(df, lista_variaveis):
    '''
    Extrai os atributos do dataframe pandas

    Params:
    -------
    df: dataframe de entrada

    Returns:
    par X, Y
    '''

    X = df[lista_variaveis]
    


    
    Y = df[['ocorrencia']]
    
    n_tempo, n_coord = conta_tempos_e_coordenadas(df)
    
    X_transformado = X.to_numpy().astype(np.float32).reshape(n_tempo , n_coord, -1)
    Y_transformado = Y.to_numpy().astype(np.float32).reshape(n_tempo , n_coord, -1)
    
    return X_transformado, Y_transformado

    
def conta_tempos_e_coordenadas(df):
    '''
    Conta o número de tempos e coordenadas de um dataframe

    Params:
    -------
    df: dataframe com os atributos 'tempo', e 'n_coord'

    Returns:
    --------
    tupla número de tempos, número de coordenadas
    '''
    tempo = len(df['tempo'].unique())
    n_coord = len(df['n_coord'].unique())

    return tempo, n_coord


def calcula_sample_weight(df,peso):
    '''
    Calcula os pesos para as amostras de treinamento
    '''
    
    Y = df[['ocorrencia']]
    sample_weight_ = list(Y['ocorrencia'])
    
    series = pd.Series(sample_weight_)
    series = series*peso #3191848/613
    series = series+1
    series = series/(peso+1)
    
    sample_weight_ = np.array(list(series)).astype(np.float32)
    
    return sample_weight_
    
    
def flatten(l):
    return [item for sublist in l for item in sublist]    

# Importacao dos dados

In [5]:
df = pd.read_csv('df_comp2.csv')

In [6]:
df.describe()

,Unnamed: 0,n_coord,ocorrencia,vintequatrohoras,Altitude_numerica,Declividade_numerica,graurisc,lon_ocr,lat_ocr,Orientacao_numerica,...,Area_Urbana,Unidades_de_Conservacao_Protecao_Integral,Plano_de_Manejo,flg_comunidades,flg_agricola,flg_exploracao_mineral,flg_rocha,flg_cobertura_vegetal,flg_afloramento_rochoso,flg_ocupacao_desordenada
count,4.501860e+06,4.501860e+06,4.501860e+06,4.501860e+06,4.501860e+06,4.501860e+06,4.501860e+06,4.501860e+06,4.501860e+06,4.501860e+06,...,4.501860e+06,4.501860e+06,4.501860e+06,4501860.0,4.501860e+06,4.501860e+06,4501860.0,4.501860e+06,4.501860e+06,4.501860e+06
mean,2.250930e+06,1.567000e+03,1.941420e-04,-7.103044e+00,4.510644e+00,3.259566e+00,-1.097688e+01,4.542922e-01,6.377735e-01,5.612156e-01,...,4.842105e-01,2.886762e-01,2.775120e-01,0.2,2.232855e-03,1.594896e-03,0.0,4.915470e-01,2.041467e-02,2.296651e-02
std,1.299575e+06,9.049966e+02,1.393213e-02,6.156592e+00,6.186039e-01,6.207251e-01,2.519012e+00,2.160416e-01,1.903145e-01,2.780649e-01,...,4.997507e-01,4.531471e-01,4.477713e-01,0.4,4.720031e-02,3.990430e-02,0.0,4.999286e-01,1.414140e-01,1.497967e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,-1.151293e+01,1.873685e+00,7.240009e-02,-1.151293e+01,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.125465e+06,7.830000e+02,0.000000e+00,-1.151293e+01,4.106021e+00,2.909836e+00,-1.151293e+01,2.810723e-01,5.094003e-01,3.573974e-01,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.250930e+06,1.567000e+03,0.000000e+00,-1.151293e+01,4.615688e+00,3.333837e+00,-1.151293e+01,4.393209e-01,6.510827e-01,5.359180e-01,...,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00
75%,3.376394e+06,2.351000e+03,0.000000e+00,1.823216e-01,4.922834e+00,3.702971e+00,-1.151293e+01,6.152376e-01,7.811985e-01,8.296920e-01,...,1.000000e+00,1.000000e+00,1.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,1.000000e+00,0.000000e+00,0.000000e+00
max,4.501859e+06,3.134000e+03,1.000000e+00,1.043501e+01,5.927562e+00,5.209883e+00,1.609438e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,1.000000e+00,1.000000e+00,1.000000e+00,1.0,1.000000e+00,1.000000e+00,0.0,1.000000e+00,1.000000e+00,1.000000e+00


In [7]:
df['tempo'].min()

'2019-02-01 09:00:00'

In [8]:
df.replace(np.nan,-1, inplace=True)

In [9]:
df['tempo'] =  pd.to_datetime(df['tempo'])

In [10]:
df = df.sort_values(['tempo','n_coord'])

In [11]:
lista_variaveis = [
     'vintequatrohoras'                                
    ,'Altitude_numerica'                               
    ,'Declividade_numerica'                          
    ,'graurisc'                                        
    ,'lon_ocr'                                         
    ,'lat_ocr'                                         
    ,'Orientacao_numerica'                             
    ,'Curv_Vertical_numerica'                          
    ,'Curv_Horizontal_numerica'                        
    ,'Relevo_sombreado_numerico'                       
    ,'Vegetacao_Natural_Dominante'                     
    ,'Area_Antropica_Dominante'                        
    ,'legenda_2_Pecuária (pastagens)'                  
    ,'Floresta_Ombrofila_Densa'                        
    ,'Formacao_Pioneira'                               
    ,'Floresta_Ombrofila_Densa_Submontana'             
    ,'Influencia_urbana'                               
    ,'Vegetacao_Secundaria'                            
    ,'Argilossolo'                                     
    ,'Gleissolo'                                       
    ,'Argilossolo_Vermelho_Amarelo'                    
    ,'Gleissolo_Melanico'                              
    ,'Area_Urbana'                                     
    ,'Unidades_de_Conservacao_Protecao_Integral'       
    ,'Plano_de_Manejo'                                 
    ,'flg_comunidades'                                 
    ,'flg_agricola'                                    
    ,'flg_exploracao_mineral'                          
    ,'flg_rocha'                                       
    ,'flg_cobertura_vegetal'                           
    ,'flg_afloramento_rochoso'                                                            
    ,'flg_ocupacao_desordenada'                         
           ]

Manter fixo:

* Teste  - 3 meses

Deslizar:

* Treinamento - 2 anos

* Validação - 6 meses

In [12]:
janela_treino = ['2019-02-01 09:00:00'
                ,'2019-03-01 09:00:00'
                ,'2019-04-01 09:00:00'
                ,'2019-05-01 09:00:00'
                ,'2019-06-01 09:00:00'
                ,'2019-07-01 09:00:00'
                ,'2019-08-01 09:00:00'
                ,'2019-09-01 09:00:00'
                ,'2019-10-01 09:00:00'
                ,'2019-11-01 09:00:00'
                ,'2019-12-01 09:00:00'
                ,'2020-01-01 09:00:00'
                ,'2020-02-01 09:00:00'
                ,'2020-03-01 09:00:00'
                ]

janela_validacao_i = ['2021-02-01 09:00:00'
                ,'2021-03-01 09:00:00'
                ,'2021-04-01 09:00:00'
                ,'2021-05-01 09:00:00'
                ,'2021-06-01 09:00:00'
                ,'2021-07-01 09:00:00'
                ,'2021-08-01 09:00:00'
                ,'2021-09-01 09:00:00'
                ,'2021-10-01 09:00:00'
                ,'2021-11-01 09:00:00'
                ,'2021-12-01 09:00:00'
                ,'2022-01-01 09:00:00'
                ,'2022-02-01 09:00:00'
                ,'2022-03-01 09:00:00'
                ]

janela_validacao_s = ['2021-08-01 09:00:00'
                ,'2021-09-01 09:00:00'
                ,'2021-10-01 09:00:00'
                ,'2021-11-01 09:00:00'
                ,'2021-12-01 09:00:00'
                ,'2022-01-01 09:00:00'
                ,'2022-02-01 09:00:00'
                ,'2022-03-01 09:00:00'
                ,'2022-04-01 09:00:00'
                ,'2022-05-01 09:00:00'
                ,'2022-06-01 09:00:00'
                ,'2022-07-01 09:00:00'
                ,'2022-08-01 09:00:00'
                ,'2022-09-01 09:00:00'
                ]


In [13]:
df_saida_modelo = pd.DataFrame({'Janela': None
                    ,'Treino Ocorrências': None
                    ,'#Treino': None
                    ,'Validação Ocorrências': None
                    ,'#Validação': None
                    ,'Teste Ocorrências': None
                    ,'#Teste': None
                    
                    ,'loss': None
                    ,'binary_crossentropy': None
                    ,'accuracy': None
                    ,'precision': None
                    ,'recall': None
                                
                    ,'Verdadeiro Negativo': None
                    ,'Falso Positivo': None
                    ,'Falso Negativo': None
                    ,'Verdadeiro Positivo': None            
                                 }, index=[0])


len(lista_variaveis)

 self.lstm = LSTM(33
                         ,activation='elu'
                         , input_shape=(time_step, input_dim,)
                         ,return_sequences=True
                         , go_backwards=True
                        , kernel_initializer="he_normal")

In [14]:
len(lista_variaveis)

32

In [15]:
class CustomModel(tf.keras.Model):

    def __init__(self, time_step, input_dim, num_classes = 2 ):
        super(CustomModel, self).__init__()
        #self.masking = Masking(mask_value=-1.,input_shape=(time_step, input_dim,))
        self.lstm = LSTM(32
                         ,activation='sigmoid'
                         , input_shape=(time_step, input_dim)
                         ,return_sequences=True
                         , go_backwards=True
                         )

        
        self.classifier = Dense(num_classes, activation='sigmoid')

    def call(self, inputs):
        x = inputs
        tf.print(" input:  \n " , inputs)
        
        #x = self.masking(inputs)
        #print(f'masking: {x}')
        #tf.print( " \n masking: \n ", x[0])
        
        x = self.lstm(x)
        print(f'lstm: {x[0]}')
        tf.print(" \n lstm: \n ", x)
        
        #x = self.lstm2(x)
        #print(f'lstm: {x[0]}')
        #tf.print(" \n lstm: \n ", x)
        
        #x = self.masking(inputs)
        #print(f'masking: {x}')
        #tf.print( " \n masking2: \n ", x[0])
        
        print(f'dense: {self.classifier(x)[0]}')
        tf.print(" \n dense: \n ", self.classifier(x))
        
        return self.classifier(x)
    
    

In [ ]:
for i in [0,1]:#range(len(janela_validacao_s)):
    print('#'*20)
    print('#'*20)
    print('Modelo 1')
    print(f' Janela {i}')
    print('#'*20)
    print('#'*20)
    
    saida_modelo = [i+1]
    
    filtro_inferior = pd.to_datetime(janela_treino[i]) <= df.loc[:, 'tempo']
    filtro_superior = df.loc[:,'tempo'] < pd.to_datetime(janela_validacao_i[i])
    treino = df[filtro_inferior & filtro_superior]
    
    filtro_inferior = pd.to_datetime(janela_validacao_i[i]) <= df.loc[:, 'tempo']
    filtro_superior = df.loc[:, 'tempo'] < pd.to_datetime(janela_validacao_s[i])
    validacao = df[filtro_inferior & filtro_superior]
    
    teste = df[df.loc[:,'tempo'] >= pd.to_datetime('2022-09-01 09:00:00')]
    
    ### Treino

    X_train, Y_train = transformacao_estrutura(treino, lista_variaveis)

    ### Validação

    X_val, Y_val = transformacao_estrutura(validacao, lista_variaveis)

    ### Teste

    X_teste, Y_teste = transformacao_estrutura(teste, lista_variaveis)
    
    
    print('Treino Ocorrências, #Treino, Validação Ocorrências, #Validação, Teste Ocorrências, #Teste')
    T_qtd_o = sum(treino['ocorrencia'])
    T_tam_o = len(treino['ocorrencia'])
    peso = T_tam_o / T_qtd_o


    V_qtd_o = sum(validacao['ocorrencia'])
    V_tam_o = len(validacao['ocorrencia'])

    Tes_qtd_o = sum(teste['ocorrencia'])
    Tes_tam_o = len(teste['ocorrencia'])

    print(f'{T_qtd_o},{T_tam_o},{V_qtd_o},{V_tam_o},{Tes_qtd_o},{Tes_tam_o}')
    
    saida_modelo.extend([T_qtd_o,T_tam_o,V_qtd_o,V_tam_o,Tes_qtd_o,Tes_tam_o])
    
    print(peso*1.3)
    
    sample_weight_ = np.asarray(calcula_sample_weight(treino,round(peso*1.5)  )).astype(np.float32)
    
    sample_weight_transformado = sample_weight_.reshape(len(treino['tempo'].unique()), len(treino['n_coord'].unique()), -1)
    
    
    ##################### Modelo ##################
    
    time_step = X_train.shape[1] # Quantidade de coordenada para equivaler a 1 espaço de tempo
    input_dim = X_train.shape[2] #qtd colunas (features)
    out = Y_train.shape[2]

    # LSTM
    start = time.time()
    model = Sequential()
    model.add(Masking(mask_value=-1.,input_shape=(time_step, input_dim,))) #camada de entrada
    model.add(LSTM(32,activation='elu', input_shape=(time_step, input_dim,),return_sequences=True, go_backwards=True)) # camada escondida
    model.add(Dense(out, activation='sigmoid')) #camada saida
    
    model = CustomModel(time_step, input_dim, out)
    
    opt = tf.keras.optimizers.Adam(learning_rate=0.1)

    model.compile(loss = tf.keras.losses.BinaryFocalCrossentropy(apply_class_balancing=True, gamma=15,label_smoothing=0.3,from_logits=False)  #https://keras.io/api/losses/probabilistic_losses/ 
                  , optimizer= opt #'adam'
                  , weighted_metrics=[tf.keras.losses.BinaryFocalCrossentropy(apply_class_balancing=True, gamma=15,label_smoothing=0.3,from_logits=False) ,'accuracy'
                  , tf.keras.metrics.Precision()
                  , tf.keras.metrics.Recall()]
                # , sample_weight_mode="temporal"   #Weights
                 )
    
    hist = model.fit(X_train
                     , Y_train
                     , epochs=50
                     ,validation_data=(X_val, Y_val)
                     , verbose=1
                     , batch_size=256
                   #  , sample_weight=sample_weight_transformado  #(samples, sequence_length)
            )
    end = time.time()
    print("Total compile time: --------", end - start, 's')
    
    predictions = (model.predict(X_teste, verbose=1) > 0.5).astype("int32").reshape((1,-1))
    
    score = model.evaluate(X_teste, Y_teste)
    
    print('loss, binary_crossentropy, accuracy, precision, recall ')
    print(f'{score}')
    
    saida_modelo.extend(score)
    
    pred_y = pd.DataFrame(flatten(predictions), columns=['Prediction']) 
    
    
    tabela=confusion_matrix(teste[['ocorrencia']],pred_y)
        
    print('Verdadeiro Negativo, Falso Positivo, Falso Negativo, Verdadeiro Positivo')
    print(f'{tabela[0,0]},{tabela[0,1]},{tabela[1,0]},{tabela[1,1]}')

    saida_modelo.extend(tabela.flatten())
    
    
    df_saida_modelo.loc[i] = saida_modelo


####################
####################
Modelo 1
 Janela 0
####################
####################
Treino Ocorrências, #Treino, Validação Ocorrências, #Validação, Teste Ocorrências, #Teste
545.0,2247795,9.0,567435,217.0,564300
5361.712844036697


2023-06-01 09:34:23.495153: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-06-01 09:34:23.496027: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-06-01 09:34:23.496163: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-06-01 09:34:23.496213: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA

Epoch 1/50
lstm: Tensor("custom_model/strided_slice:0", shape=(3135, 32), dtype=float32)
dense: Tensor("custom_model/strided_slice_1:0", shape=(3135, 1), dtype=float32)
lstm: Tensor("custom_model/strided_slice:0", shape=(3135, 32), dtype=float32)
dense: Tensor("custom_model/strided_slice_1:0", shape=(3135, 1), dtype=float32)
 input:  
  [[[-0.91629076 4.24826956 3.70569849 ... 1 0 0]
  [-0.91629076 3.60378599 3.63325787 ... 1 0 0]
  [-0.91629076 4.3122611 3.37637687 ... 1 0 0]
  ...
  [-0.91629076 4.7226882 2.89473248 ... 0 0 0]
  [-0.223143548 4.48119879 3.54951859 ... 0 0 0]
  [-1.60943794 4.00240517 2.94308448 ... 0 0 0]]

 [[-11.5129251 4.24826956 3.70569849 ... 1 0 0]
  [2.56494927 3.60378599 3.63325787 ... 1 0 0]
  [2.56494927 4.3122611 3.37637687 ... 1 0 0]
  ...
  [2.94443893 4.7226882 2.89473248 ... 0 0 0]
  [3.07269335 4.48119879 3.54951859 ... 0 0 0]
  [2.8903718 4.00240517 2.94308448 ... 0 0 0]]

 [[-11.5129251 4.24826956 3.70569849 ... 1 0 0]
  [-11.5129251 3.60378599 3.63

2023-06-01 09:34:25.772810: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


 
 lstm: 
  [[[0.298588634 0.523624539 0.539106 ... 0.480417609 0.581972837 0.469960302]
  [0.306576878 0.505847514 0.389142931 ... 0.40791747 0.387375325 0.300394773]
  [0.411731035 0.569000423 0.621696234 ... 0.535186589 0.648120642 0.581926048]
  ...
  [0.48145017 0.552221656 0.847816408 ... 0.612539291 0.583347261 0.597304523]
  [0.462765545 0.549266 0.847147703 ... 0.613550782 0.573043883 0.623360395]
  [0.459786445 0.556894064 0.855695188 ... 0.615710139 0.579645813 0.6126616]]

 [[0.26624015 0.494194806 0.494309694 ... 0.387359142 0.608727217 0.541780531]
  [0.278188378 0.4678334 0.3283346 ... 0.33558163 0.456606418 0.402616233]
  [0.385213345 0.531302214 0.543348253 ... 0.446434081 0.66169554 0.658859193]
  ...
  [0.478976816 0.524634123 0.713882089 ... 0.533465624 0.600280106 0.751610935]
  [0.458480567 0.52165854 0.714724541 ... 0.534423113 0.592954159 0.77129221]
  [0.483604163 0.623442 0.841528177 ... 0.835003793 0.433937162 0.178116292]]

 [[0.368403822 0.602913201 0.64806

...

 
 dense: 
  [[[0.32201]
  [0.299374044]
  [0.301506519]
  ...
  [0.293557227]
  [0.29510504]
  [0.293763608]]

 [[0.32201]
  [0.299374044]
  [0.301506519]
  ...
  [0.293557227]
  [0.29510504]
  [0.293763608]]

 [[0.340741187]
  [0.351561606]
  [0.315712422]
  ...
  [0.316391259]
  [0.319324374]
  [0.316031426]]

 ...

 [[0.326431692]
  [0.300218254]
  [0.302102089]
  ...
  [0.293557227]
  [0.29510504]
  [0.293763608]]

 [[0.359556943]
  [0.369213194]
  [0.332483023]
  ...
  [0.333402872]
  [0.33539781]
  [0.331242889]]

 [[0.32201]
  [0.299374044]
  [0.301506519]
  ...
  [0.293557227]
  [0.29510504]
  [0.293763608]]]
2/3 [===================>..........] - ETA: 0s - loss: 1.6177e-07 - binary_focal_crossentropy: 1.6177e-07 - accuracy: 0.9997 - precision_1: 0.0000e+00 - recall_1: 0.0000e+00 input:  
  [[[-11.5129251 4.24826956 3.70569849 ... 1 0 0]
  [-11.5129251 3.60378599 3.63325787 ... 1 0 0]
  [-11.5129251 4.3122611 3.37637687 ... 1 0 0]
  ...
  [-11.5129251 4.7226882 2.89473248 ... 

 
 dense: 
  [[[0.324092984]
  [0.339754552]
  [0.303306609]
  ...
  [0.296099871]
  [0.297688395]
  [0.296326637]]

 [[0.324092984]
  [0.301566809]
  [0.303657115]
  ...
  [0.296099871]
  [0.297688395]
  [0.296326637]]

 [[0.324092984]
  [0.301566809]
  [0.303657115]
  ...
  [0.296099871]
  [0.297688395]
  [0.296326637]]

 ...

 [[0.324092984]
  [0.301566809]
  [0.303657115]
  ...
  [0.296099871]
  [0.297688395]
  [0.296326637]]

 [[0.340129256]
  [0.342915952]
  [0.317610919]
  ...
  [0.323021799]
  [0.325802803]
  [0.324058145]]

 [[0.354131758]
  [0.362597436]
  [0.328430116]
  ...
  [0.329916298]
  [0.332316548]
  [0.32842046]]]
3/3 [==============================] - 3s 942ms/step - loss: 1.5602e-07 - binary_focal_crossentropy: 1.5509e-07 - accuracy: 0.9998 - precision_1: 0.0000e+00 - recall_1: 0.0000e+00 - val_loss: 6.5460e-08 - val_binary_focal_crossentropy: 6.5460e-08 - val_accuracy: 1.0000 - val_precision_1: 0.0000e+00 - val_recall_1: 0.0000e+00
Epoch 30/50
 input:  
  [[[-11.

 
 lstm: 
  [[[0.00227829232 0.0388124026 0.0056052357 ... 0.0335931927 0.488506 0.0338850655]
  [0.118613422 0.164480537 0.0238930155 ... 0.0320530497 0.480943501 0.0142907547]
  [0.00231866818 0.0412437655 0.00610957528 ... 0.0411995 0.490271419 0.0296653919]
  ...
  [0.00347242458 0.0506985448 0.0106142377 ... 0.0535501763 0.489542276 0.0375042111]
  [0.0038413275 0.057762403 0.00931927282 ... 0.0520834848 0.487254322 0.0458797738]
  [0.00411043223 0.0564557202 0.00958679058 ... 0.0530403554 0.488482833 0.0415261]]

 [[0.00227829232 0.0388124026 0.0056052357 ... 0.0335931927 0.488506 0.0338850655]
  [0.118613422 0.164480537 0.0238930155 ... 0.0320530497 0.480943501 0.0142907547]
  [0.00231866818 0.0412437655 0.00610957528 ... 0.0411995 0.490271419 0.0296653919]
  ...
  [0.00347242458 0.0506985448 0.0106142377 ... 0.0535501763 0.489542276 0.0375042111]
  [0.0038413275 0.057762403 0.00931927282 ... 0.0520834848 0.487254322 0.0458797738]
  [0.00411043223 0.0564557202 0.00958679058 ... 

 
 dense: 
  [[[0.323922932]
  [0.301771164]
  [0.303484321]
  ...
  [0.29611966]
  [0.297744]
  [0.296363771]]

 [[0.35179016]
  [0.34310928]
  [0.330186039]
  ...
  [0.332566559]
  [0.33494705]
  [0.327677906]]

 [[0.324911565]
  [0.333075464]
  [0.301894069]
  ...
  [0.304126263]
  [0.30776611]
  [0.304934114]]

 ...

 [[0.352668852]
  [0.348721415]
  [0.313484]
  ...
  [0.316222966]
  [0.319449365]
  [0.312681]]

 [[0.327863306]
  [0.328223854]
  [0.304209471]
  ...
  [0.29611966]
  [0.297744]
  [0.296363771]]

 [[0.323922932]
  [0.301771164]
  [0.303484321]
  ...
  [0.29611966]
  [0.297744]
  [0.296363771]]]
3/3 [==============================] - ETA: 0s - loss: 1.5595e-07 - binary_focal_crossentropy: 1.5582e-07 - accuracy: 0.9998 - precision_1: 0.0000e+00 - recall_1: 0.0000e+00 input:  
  [[[-11.5129251 4.24826956 3.70569849 ... 1 0 0]
  [-11.5129251 3.60378599 3.63325787 ... 1 0 0]
  [-11.5129251 4.3122611 3.37637687 ... 1 0 0]
  ...
  [-11.5129251 4.7226882 2.89473248 ... 0 0 0

 
 lstm: 
  [[[0.00225065392 0.0387882479 0.00551478891 ... 0.0336742178 0.488709182 0.0338105373]
  [0.117891647 0.16439575 0.0236742292 ... 0.0321144201 0.481338531 0.0142573472]
  [0.00229099905 0.0412145816 0.00600410532 ... 0.0413195267 0.490446627 0.0295961946]
  ...
  [0.00342847197 0.0506425686 0.0104483934 ... 0.0537117347 0.489670724 0.0374421]
  [0.00379297975 0.057698641 0.00917719863 ... 0.0522391424 0.487403423 0.0458088629]
  [0.00405825861 0.0563927367 0.00943747815 ... 0.0532013215 0.488620698 0.0414598137]]

 [[0.0112777939 0.0744080618 0.111733168 ... 0.192412436 0.389339447 0.115343601]
  [0.377430201 0.275267839 0.432479531 ... 0.25347665 0.250511169 0.078373678]
  [0.0144496979 0.0821654648 0.214110732 ... 0.267947078 0.362208277 0.145161465]
  ...
  [0.0218446944 0.0992186591 0.277507514 ... 0.317272931 0.350348562 0.1699581]
  [0.0241771378 0.111944102 0.256999344 ... 0.313914388 0.327478319 0.19620423]
  [0.0243600924 0.106619425 0.240235507 ... 0.305539548 0.3

In [ ]:
df_saida_modelo.to_csv('Modelo2.csv')